In [14]:
from datasets.cross_tissue_atlas import CrossTissueDataset
import os

import hydra
from omegaconf import OmegaConf

import torch
import numpy as np

from geomloss import SamplesLoss

In [15]:
train_set = CrossTissueDataset(
    root="data",
    split="train",
)

test_set = CrossTissueDataset(
    root="data",
    split="test",
)

In [16]:
adata = test_set.adata

In [17]:
train_donors = train_set.donors
test_donors = test_set.donors
print(train_donors)
print(test_donors)

['GTEX-12BJ1', 'GTEX-13N11', 'GTEX-144GM', 'GTEX-145ME', 'GTEX-15CHR', 'GTEX-15EOM', 'GTEX-15RIE', 'GTEX-15SB6']
['GTEX-16BQI', 'GTEX-1CAMR', 'GTEX-1CAMS', 'GTEX-1HSMQ', 'GTEX-1I1GU', 'GTEX-1ICG6', 'GTEX-1MCC2', 'GTEX-1R9PN']


In [18]:
donor_centroids = {}
for donor in test_donors:
    donor_data = test_set.adata[test_set.adata.obs['donor_id'] == donor]
    centroid = np.nan_to_num(donor_data.obsm['X_pca']).mean(axis=0).flatten()
    donor_centroids[donor] = centroid
for donor in train_donors:
    donor_data = train_set.adata[train_set.adata.obs['donor_id'] == donor]
    centroid = np.nan_to_num(donor_data.obsm['X_pca']).mean(axis=0).flatten()
    donor_centroids[donor] = centroid

# for each test donor, get the nearest train donor
test_to_train_donor = {}
for test_donor in test_donors:
    test_centroid = donor_centroids[test_donor]
    nearest_train_donor = None
    nearest_distance = float('inf')
    for train_donor in train_donors:
        train_centroid = donor_centroids[train_donor]
        distance = torch.norm(torch.tensor(test_centroid) - torch.tensor(train_centroid)).item()
        if distance < nearest_distance:
            nearest_distance = distance
            nearest_train_donor = train_donor
    test_to_train_donor[test_donor] = nearest_train_donor

print("Test to Train Donor Mapping:")
for test_donor, train_donor in test_to_train_donor.items():
    print(f"{test_donor} -> {train_donor}")

Test to Train Donor Mapping:
GTEX-16BQI -> GTEX-15SB6
GTEX-1CAMR -> GTEX-13N11
GTEX-1CAMS -> GTEX-15EOM
GTEX-1HSMQ -> GTEX-13N11
GTEX-1I1GU -> GTEX-12BJ1
GTEX-1ICG6 -> GTEX-13N11
GTEX-1MCC2 -> GTEX-15EOM
GTEX-1R9PN -> GTEX-13N11


In [19]:
output_dir = '/orcd/data/omarabu/001/gokul/CoupledDistributionEmbeddings/outputs/'

configs = os.listdir(output_dir)

config_name = [x for x in configs if x.startswith('crosstissue_gnn_energy_')][0]
print(config_name)
config_path = os.path.join(output_dir, config_name, 'config.yaml')
if not os.path.exists(config_path):
    raise FileNotFoundError(f"Config not found at {config_path}")

config = OmegaConf.load(config_path)

# Detect model types
encoder_type, generator_type = ('esm', 'progen2')

best_model_path = os.path.join(output_dir, config_name, 'best_model.pt')
if not os.path.exists(best_model_path):
    raise FileNotFoundError(f"Best model not found at {best_model_path}")

encoder = hydra.utils.instantiate(config.encoder)
generator = hydra.utils.instantiate(config.generator)

device = 'cuda'
checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)

encoder.load_state_dict(checkpoint['encoder_state_dict'])
generator.load_state_dict(checkpoint['generator_state_dict'])

epoch = checkpoint.get('epoch', 'unknown')
loss = checkpoint.get('loss', float('nan'))

print(epoch, loss)

encoder.to(device)
generator.to(device)
encoder.eval()
generator.eval();

crosstissue_gnn_energy_a0461089364e592ec6d81805f7ff3038
4800 3.937213897705078


In [20]:
energy = SamplesLoss("energy")

energy_dists = []

for p in range(8):

    for _ in range(100):

        batch = test_set[p]

        source_samples = batch['source_samples'].to(device).unsqueeze(0)
        target_samples = batch['target_samples'].to(device).unsqueeze(0)

        with torch.no_grad():
            source_latent = encoder(source_samples)
            target_latent = encoder(target_samples)

            # print(source_latent.shape)

            samples = generator.sample(source_samples.reshape(-1, 50), source_latent, target_latent)

        e_dist = energy(samples.squeeze(0), target_samples.squeeze(0)).item()
        energy_dists.append(e_dist)

print("mean Energy Distance:", np.mean(energy_dists))
print('s.e.m', np.std(energy_dists) / np.sqrt(len(energy_dists)))

mean Energy Distance: 0.8893541198968887
s.e.m 0.018888060575417576


In [21]:
output_dir = '/orcd/data/omarabu/001/gokul/CoupledDistributionEmbeddings/outputs/'

configs = os.listdir(output_dir)

config_name = [x for x in configs if x.startswith('crosstissue_onehot_energy_')][0]
print(config_name)
config_path = os.path.join(output_dir, config_name, 'config.yaml')
if not os.path.exists(config_path):
    raise FileNotFoundError(f"Config not found at {config_path}")

config = OmegaConf.load(config_path)

# Detect model types
encoder_type, generator_type = ('esm', 'progen2')

best_model_path = os.path.join(output_dir, config_name, 'best_model.pt')
if not os.path.exists(best_model_path):
    raise FileNotFoundError(f"Best model not found at {best_model_path}")

encoder = hydra.utils.instantiate(config.encoder)
generator = hydra.utils.instantiate(config.generator)

device = 'cuda'
checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)

encoder.load_state_dict(checkpoint['encoder_state_dict'])
generator.load_state_dict(checkpoint['generator_state_dict'])

epoch = checkpoint.get('epoch', 'unknown')
loss = checkpoint.get('loss', float('nan'))

print(epoch, loss)

encoder.to(device)
generator.to(device)
encoder.eval()
generator.eval();

crosstissue_onehot_energy_72d66fad894ab5988deff2e8a63d3adf
4800 3.9537558555603027


In [22]:
energy = SamplesLoss("energy")

energy_dists = []

for p in range(8):

    for _ in range(100):

        batch = test_set[p]

        source_samples = batch['source_samples'].to(device).unsqueeze(0)
        target_samples = batch['target_samples'].to(device).unsqueeze(0)

        source_donor = batch['source_metadata']['donor_id']
        target_donor = batch['target_metadata']['donor_id']

        nearest_source_donor = test_to_train_donor[source_donor]
        nearest_target_donor = test_to_train_donor[target_donor]

        nn_source_idx = torch.tensor(train_set.donors.index(nearest_source_donor)).cuda()
        nn_target_idx = torch.tensor(train_set.donors.index(nearest_target_donor)).cuda()

        with torch.no_grad():
            source_latent = encoder(nn_source_idx).repeat(source_samples.shape[0], 1)
            target_latent = encoder(nn_target_idx).repeat(target_samples.shape[0], 1)

            # print(source_latent.shape)

            samples = generator.sample(source_samples.reshape(-1, 50), source_latent, target_latent)

        e_dist = energy(samples.squeeze(0), target_samples.squeeze(0)).item()
        energy_dists.append(e_dist)

print("mean Energy Distance:", np.mean(energy_dists))
print('s.e.m', np.std(energy_dists) / np.sqrt(len(energy_dists)))

mean Energy Distance: 1.1467216444015502
s.e.m 0.023265661657707454
